In [2]:
from modeling_distillemb import BertModel, BertForSequenceClassification
from distill_emb import DistillEmbSmall, DistillEmb
from config import DistillModelConfig, DistillEmbConfig
import torch
from transformers import AutoTokenizer, RwkvConfig, RwkvModel, AutoModel
from tokenizer import CharTokenizer
from knn_classifier import KNNTextClassifier
from data_loader import load_sentiment, load_ner_dataset, load_pos_dataset
from data_loader import load_news_dataset
import pandas as pd
from retrieval import build_json_pairs, top1_accuracy
import os
from transformers import GPT2LMHeadModel

In [3]:
num_input_chars=12

In [4]:
tokenizer = CharTokenizer.from_pretrained(pretrained_directory="distil-emb-base")
distill_config = DistillEmbConfig.from_pretrained(pretrained_model_name_or_path="distil-emb-base")
distill_model = DistillEmb.from_pretrained(pretrained_model_name_or_path="distil-emb-base")

In [5]:
distill_config

DistillEmbConfig {
  "activation": "gelu",
  "architectures": [
    "DistillEmb"
  ],
  "char_vocab_size": 1518,
  "distill_dropout": 0.0,
  "dtype": "float32",
  "embedding_size": 512,
  "model_type": "distilemb",
  "num_input_chars": 12,
  "pad_char_id": 0,
  "size": "base",
  "transformers_version": "4.57.3",
  "use_normalize": false,
  "use_tanh": false
}

In [6]:
# distill_config.distill_dropout = 0.25
config = DistillModelConfig(
    vocab_size=30522,
    hidden_size=512,
    num_hidden_layers=1,
    num_attention_heads=8,
    intermediate_size=3072,
    max_position_embeddings=1024,
    type_vocab_size=2,
    pad_token_id=0,
    position_embedding_type="absolute",
    use_cache=True,
    classifier_dropout=None,
    hidden_dropout_prob=0.1,
    embedding_type="distill",  # 'distilemb', 'fasttext'
    encoder_type='lstm', #'lstm'
    num_input_chars=num_input_chars,  # number of characters in each token
    char_vocab_size=tokenizer.char_vocab_size,
    distill_config=distill_config,
    distill_pretrained_model_name="distil-emb-base",
    is_decoder=False
)


In [7]:
df = pd.read_parquet("downstream-data/masakhanews.parquet")

In [8]:
df

,label,headline,text,headline_text,url,lang,split
0,5,የስፖርት ኮከቦች እና የንግድ ምልክቶቻቸው- ከቦልት እስከ ክርስቲያኖ ሮናልዶ,የአትሌቲክሱ ዓለም ኮከብ እና ፈጣኑ ሰው ዩሴን ቦልት ከውድድር በፊት እና...,የስፖርት ኮከቦች እና የንግድ ምልክቶቻቸው- ከቦልት እስከ ክርስቲያኖ ሮና...,https://www.bbc.com/amharic/articles/ceknk30j2xxo,amh,train
1,5,እግር ኳስ፡ ዩናይትድ፣ አርሴናል፣ ቼልሲ . . . ምን አስበዋል?,የስፖርት ጋዜጦች ስለ እግር ኳስ ምን እያሉ ነው? በሚቀጥለው ጥር የሚከፈ...,እግር ኳስ፡ ዩናይትድ፣ አርሴናል፣ ቼልሲ . . . ምን አስበዋል? የስፖር...,https://www.bbc.com/amharic/news-55435608,amh,train
2,0,ዓለምን ካስጨነቃት የዋጋ ንረት ተጠቃሚዎቹ እነማን ናቸው?,ከኮሮናቫይረስ ወረርሽኝ ተጽእኖ ሳያገግም የዩክሬን እና ሩሲያ ጦርነት የገ...,ዓለምን ካስጨነቃት የዋጋ ንረት ተጠቃሚዎቹ እነማን ናቸው? ከኮሮናቫይረስ ...,https://www.bbc.com/amharic/articles/crgj31l8mzzo,amh,train
3,2,ኮሮናቫይረስ፡ በቫይረሱ የሞቱት የሮማኒያው ከንቲባ በምርጫ አሸነፉ,በኮሮናቫይረስ የሞቱት የሮማኒያው ከንቲባ በቅርቡ የተደረገውን ምርጫ በከፍ...,ኮሮናቫይረስ፡ በቫይረሱ የሞቱት የሮማኒያው ከንቲባ በምርጫ አሸነፉ በኮሮና...,https://www.bbc.com/amharic/news-54336166,amh,train
4,2,ኮሮናቫይረስ፡ አውሮፕላኖች እንዴት ነው በፀረ- ተህዋሲያን የሚፀዱት?,የኮሮናቫይረስ ወረርሽኝ መከሰቱን ተከትሎ ቀጥ ብሎ የነበረውን የአለም የ...,ኮሮናቫይረስ፡ አውሮፕላኖች እንዴት ነው በፀረ- ተህዋሲያን የሚፀዱት? የኮ...,https://www.bbc.com/amharic/53627279,amh,train
...,...,...,...,...,...,...,...
30804,3,Ìròyìn ẹlẹ́jẹ̀ ni pé mo buwọ́lu Tinubu fún ipò...,"Aarẹ orilẹede Naijiria nigba kan ri, Oloye Olu...",Ìròyìn ẹlẹ́jẹ̀ ni pé mo buwọ́lu Tinubu fún ipò...,https://www.bbc.com/yoruba/articles/c2vp92lge0wo,yor,dev
30805,1,Nollywood Yoruba epic movies: Wo àwọn sinimá m...,Ọpọlọpọ awọn ere itage ati sinima lawọn agba o...,Nollywood Yoruba epic movies: Wo àwọn sinimá m...,https://www.bbc.com/yoruba/afrika-55751644,yor,dev
30806,3,"Osun: Gboyega Oyetola, Adeleke ń ṣe àríyànjiyà...",Ọrọ naa bẹrẹ pẹlu gomina Gboyega Oyetola to ni...,"Osun: Gboyega Oyetola, Adeleke ń ṣe àríyànjiyà...",https://www.bbc.com/yoruba/afrika-49058203,yor,dev
30807,5,Tokyo 2020 Olympics: Àwọn eléré ìdárayá ilẹ̀ A...,Odun 2020 lo yẹ ki idije ere idaraya Olimpiki ...,Tokyo 2020 Olympics: Àwọn eléré ìdárayá ilẹ̀ A...,https://www.bbc.com/yoruba/57936117,yor,dev


In [9]:
len(df['lang'].unique())

16

In [10]:
num_labels = len(df['label'].unique())
config.num_labels = num_labels
model = BertForSequenceClassification(config)

In [11]:
labels = [x.item() for x in df['label'].unique()]
print(labels)

[5, 0, 2, 3, 1, 6, 4]


In [12]:
from datasets import Dataset, DatasetDict
df['text'] = df['headline_text']
# Assuming df is your dataframe
# Split the data based on the 'split' column
train_df = df[df['split'] == 'train'][['text', 'label']]
test_df = df[df['split'] == 'test'][['text', 'label']]

# Create HuggingFace datasets
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [13]:
train_dataset

Dataset({
    features: ['text', 'label', '__index_level_0__'],
    num_rows: 21590
})

In [14]:
from typing import Dict, Any

def preprocess_function(examples: Dict[str, Any]):
    batch = tokenizer(
        examples["text"],
        padding=False,
        max_length=256,
        return_attention_mask=False,
    )

    batch["labels"] = examples["label"]
    return batch



tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/21590 [00:00<?, ? examples/s]

Map:   0%|          | 0/6168 [00:00<?, ? examples/s]

In [15]:
len(train_dataset[0]['text'].split())

293

In [16]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        batch = self.tokenizer.pad(
            features,
            padding="longest",
            max_length=256,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        return batch

data_collator = CustomDataCollator(tokenizer)

In [17]:
##### from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted', labels=labels)
    f1_macro = f1_score(labels, predictions, average='macro', labels=labels)
    f1_micro = f1_score(labels, predictions, average='micro', labels=labels)
    return {"accuracy": acc, "f1_weighted": f1, "f1_macro": f1_macro, "f1_micro": f1_micro}


import os
dataloader_num_workers=os.cpu_count() - 1
batch_size = 32

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=1e-4,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=15,
    weight_decay=0.01,
    report_to=[],
    eval_strategy="epoch",  
    save_total_limit=1,
    save_only_model=True,
    logging_strategy="steps",
    logging_steps=10,
    label_smoothing_factor=0.1,
    max_grad_norm=5.0,
    warmup_ratio=0.0,
    lr_scheduler_type="cosine",
    dataloader_num_workers=16,        # Number of CPU workers for data loading
    dataloader_pin_memory=True,      # Faster GPU transfer
    gradient_accumulation_steps=4
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

# Evaluate the model after training
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

AttributeError: 'list' object has no attribute 'dim'

In [ ]:
trainer.evaluate()

{'eval_loss': 0.7456780076026917,
 'eval_accuracy': 0.8764591439688716,
 'eval_f1_weighted': 0.8885552398664728,
 'eval_f1_macro': 0.875707142699436,
 'eval_f1_micro': 0.8890374291937203,
 'eval_runtime': 10.6705,
 'eval_samples_per_second': 578.045,
 'eval_steps_per_second': 18.087,
 'epoch': 15.0}

In [ ]:
# model = BertForSequenceClassification.from_pretrained("distil-emb-seqcls-lstm").cuda()
model = trainer.model
model.eval()
# Ensure 'language' column exists in df
test_df = df[df['split'] == 'test'][['text', 'label', 'lang']]
languages = test_df['lang'].unique()
per_language_f1 = {}

batch_size = 16

for lang in languages:
    lang_df = test_df[test_df['lang'] == lang]
    texts = lang_df['text'].tolist()
    labels = lang_df['label'].values
    preds = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_labels = labels[i:i+batch_size]
        tokenized = tokenizer(
            batch_texts,
            padding='longest',
            truncation=True,
            max_length=256,
            return_tensors="pt",
            return_attention_mask=True,
            padding_side="right"
        )
        with torch.no_grad():
            inputs = {k: v.cuda() for k, v in tokenized.items()}
            outputs = model(**inputs)
            batch_preds = outputs.logits.argmax(dim=-1).cpu().numpy()
            preds.extend(batch_preds)
    f1 = f1_score(labels, preds, average='weighted', labels=labels)
    per_language_f1[lang] = f1

# Print per-language F1
for lang, f1 in per_language_f1.items():
    print(f"Language: {lang}, F1: {f1:.4f}")

# Average F1
average_f1 = sum(per_language_f1.values()) / len(per_language_f1)
print(f"Average F1 across languages: {average_f1:.4f}")

Language: amh, F1: 0.9151
Language: eng, F1: 0.8971
Language: fra, F1: 0.8710
Language: hau, F1: 0.8983
Language: ibo, F1: 0.8607
Language: lin, F1: 0.8763
Language: lug, F1: 0.8655
Language: orm, F1: 0.9089
Language: pcm, F1: 0.9625
Language: run, F1: 0.9201
Language: sna, F1: 0.9093
Language: som, F1: 0.8134
Language: swa, F1: 0.8457
Language: tir, F1: 0.8507
Language: xho, F1: 0.9235
Language: yor, F1: 0.9158
Average F1 across languages: 0.8896


In [ ]:
model.save_pretrained("distil-emb-news-lstm-best-256")